In [3]:
%%writefile jurassic.py
import multiprocessing
import random
import time

# Constants
MIN_VALUE = 0

# Critical events
CRITICAL_EVENTS = {
    "T-Rex outside enclosure",
    "Electric fence failure",
    "Communication loss",
    "Security alert",
    "Unauthorized access",
}

# Zones and their events with probabilities
ZONES = {
    "T-Rex Sector": [
        ("All clear",               80),
        ("T-Rex outside enclosure", 10),
        ("Electric fence failure",  10),
    ],
    "Velociraptor Area": [
        ("All clear",              70),
        ("Visibility loss",        20),
        ("Electric fence failure", 10),
    ],
    "Triceratops Enclosure": [
        ("All clear",        60),
        ("Unusual behavior", 30),
        ("Stampede",         10),
    ],
    "Visitor Center": [
        ("All clear",          80),
        ("Communication loss", 15),
        ("Security alert",      5),
    ],
    "Genetic Laboratory": [
        ("All clear",           80),
        ("System failure",      10),
        ("Communication loss",   5),
        ("Unauthorized access",  5),
    ],
}


def get_random_event(events):
    names   = [e[0] for e in events]
    weights = [e[1] for e in events]
    return random.choices(names, weights=weights)[0]


def monitor(zone, events, duration, frequency, lock):
    total_events   = 0
    total_critical = 0
    start_time     = time.monotonic()

    while time.monotonic() - start_time < duration:
        event = get_random_event(events)
        total_events += 1

        if event in CRITICAL_EVENTS:
            total_critical += 1

        with lock:
            print(f"[{zone}] >> {event}", flush=True)

        elapsed   = time.monotonic() - start_time
        remaining = duration - elapsed

        if remaining <= MIN_VALUE:
            break

        time.sleep(min(frequency, remaining))

    with lock:
        print()
        print(f"========== {zone} ==========")
        print(f"Events   : {total_events}")
        print(f"Critical : {total_critical}")
        print()


def main():
    # Parameters - change these values as needed
    DURATION  = 20.0  # seconds
    FREQUENCY = 3.0   # seconds

    if DURATION <= MIN_VALUE or FREQUENCY <= MIN_VALUE:
        print("Duration and frequency must be greater than 0.")
        return

    lock      = multiprocessing.Lock()
    processes = []

    for zone, events in ZONES.items():
        p = multiprocessing.Process(
            target=monitor,
            args=(zone, events, DURATION, FREQUENCY, lock)
        )
        processes.append(p)

    for p in processes:
        p.start()

    for p in processes:
        p.join()

    print("===================================")
    print("All zones monitored successfully.")
    print("===================================")


if __name__ == "__main__":
    main()

Writing jurassic.py


In [4]:
!python jurassic.py

[T-Rex Sector] >> All clear
[Velociraptor Area] >> All clear
[Triceratops Enclosure] >> All clear
[Visitor Center] >> All clear
[Genetic Laboratory] >> System failure
[T-Rex Sector] >> T-Rex outside enclosure
[Velociraptor Area] >> All clear
[Triceratops Enclosure] >> Stampede
[Visitor Center] >> All clear
[Genetic Laboratory] >> All clear
[T-Rex Sector] >> All clear
[Velociraptor Area] >> All clear
[Triceratops Enclosure] >> Unusual behavior
[Visitor Center] >> All clear
[Genetic Laboratory] >> All clear
[T-Rex Sector] >> All clear
[Velociraptor Area] >> Visibility loss
[Triceratops Enclosure] >> All clear
[Genetic Laboratory] >> All clear
[Visitor Center] >> All clear
[T-Rex Sector] >> All clear
[Velociraptor Area] >> All clear
[Triceratops Enclosure] >> All clear
[Genetic Laboratory] >> System failure
[Visitor Center] >> All clear
[T-Rex Sector] >> All clear
[Velociraptor Area] >> All clear
[Triceratops Enclosure] >> Unusual behavior
[Visitor Center] >> All clear
[Genetic Laboratory

Conclusiones

En esta actividad creamos un programa concurrente en Python donde cada zona del parque corre como un proceso independiente al mismo tiempo.

Utilizamos multiprocessing ya que son procesos pesados, cada uno con su propio espacio de memoria.

Como todos los procesos imprimen en la misma consola, tuvimos que usar un Lock para que no se pisen entre sí. El lock se pasa como argumento a cada proceso.

También se puede ver el no determinismo típico de la concurrencia — el orden en que aparecen los eventos cambia cada vez que se ejecuta el programa, dependiendo de cómo el sistema operativo reparte el tiempo entre los procesos.

Al finalizar, cada proceso muestra su propio resumen con los eventos detectados y los críticos.